# Getting Started with Typeflux

This notebook introduces Typeflux without project structure. You will define schemas, prompts, a provider, and a workflow inline so the moving parts are visible.

In [ ]:
from __future__ import annotations

from enum import StrEnum
from pathlib import Path
import os

from pydantic import BaseModel, Field
from typeflux import InlineResolver, PromptRef, ResolvedPrompt, StepSpec, WorkflowSpec, run_workflow
from typeflux.providers.base import ModelProvider
from typeflux.providers.openai import OpenAIProvider

ROOT = Path.cwd()
if not (ROOT / ".env.example").exists():
    ROOT = Path.cwd().parent

def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        if not line or line.strip().startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

load_dotenv(ROOT / ".env")

## 1. Define the data contract

Typeflux uses pydantic models as the boundary between each step. The LLM is asked to return an instance of the output model, and Typeflux validates it before the next step receives it. Notice that `topic` is an enum, so downstream code receives one of the known buckets rather than arbitrary text.

In [ ]:
class NoteTopic(StrEnum):
    BILLING = "billing"
    SUPPORT = "support"
    ACCOUNT = "account"
    SALES = "sales"
    OTHER = "other"


class NoteInput(BaseModel):
    note: str = Field(description="Raw note from a user or teammate.")


class NoteSummary(BaseModel):
    topic: NoteTopic = Field(description="Controlled topic bucket.")
    summary: str = Field(description="One sentence summary.")
    action_required: bool = Field(description="Whether someone needs to act.")


sample = NoteInput(note="Dana asked whether the invoice can be resent before Friday.")
sample

## 2. Write the prompt text

Keeping the prompt in a named `PROMPT_TEXT` constant makes the next cell easier to read: one cell authors the prompt, the next cell registers it.

In [ ]:
PROMPT_TEXT = """
You are helping triage short internal notes.

Classify the note into exactly one topic:
- billing: invoices, receipts, refunds, charges, payment questions
- support: bugs, outages, technical help, troubleshooting
- account: login, permissions, profile, security, access
- sales: pricing, upgrades, seats, procurement, demos
- other: anything that does not fit the buckets above

Then write a one-sentence summary and decide whether a human needs to take action.

Note:
{{note}}
""".strip()

## 3. Add a prompt resolver

A workflow step stores a `PromptRef`. A resolver turns that reference into prompt messages. For the first pass, use `InlineResolver` so there is no external registry involved.

In [ ]:
prompt_ref = PromptRef("note-summary")
resolver = InlineResolver(
    prompts={
        (prompt_ref.name, prompt_ref.version): ResolvedPrompt.from_text(
            prompt_ref,
            PROMPT_TEXT,
            template_format="mustache",
        )
    }
)

## 4. Use a deterministic provider first

The provider is the only thing that talks to an LLM. During development and tests, a fake provider lets you exercise the workflow contract without spending tokens or depending on network access.

In [ ]:
class FakeProvider(ModelProvider):
    def structured_call(self, *, messages, output_schema, model=None, temperature=None):
        rendered_prompt = messages[0].content
        return output_schema(
            topic=NoteTopic.BILLING,
            summary="Dana needs the invoice resent before Friday.",
            action_required="invoice" in rendered_prompt.lower(),
        )


fake_provider = FakeProvider()

## 5. Build and run a workflow

A `StepSpec` says: given this input schema, ask this prompt, validate this output schema. A `WorkflowSpec` chains one or more steps.

In [ ]:
summarize_note = StepSpec(
    name="summarize_note",
    input_type=NoteInput,
    output_type=NoteSummary,
    prompt_ref=prompt_ref,
)

workflow = WorkflowSpec(name="GettingStarted", steps=[summarize_note])

result = run_workflow(workflow, sample, resolver=resolver, provider=fake_provider)
result

## 6. Optional live run

Once `OPENAI_API_KEY` is set in the root `.env`, swap only the provider. The schemas, prompt, and workflow stay the same.

In [ ]:
if os.getenv("OPENAI_API_KEY"):
    live_provider = OpenAIProvider(default_model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"))
    live_result = run_workflow(workflow, sample, resolver=resolver, provider=live_provider)
    display(live_result)
else:
    print("Set OPENAI_API_KEY in the root .env to run this cell live.")